# 101 · Engineering perspective mini lab

This notebook goes with the article
[Engineering perspective](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/101/engineer-perspective/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/101/engineering_perspective.ipynb)

Services need a short decision path before anyone opens a benchmark chart.
You will classify a few situations, encode the same request-shaped object several ways,
and see why “it parsed as JSON” is not the same as “it is a valid request.”

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


In [ ]:
import json

# Public-ish API DTO
DTO = {
    "request_id": "req-7f3a",
    "user_id": 42,
    "items": [{"sku": "A-1", "qty": 2}, {"sku": "B-9", "qty": 1}],
    "metadata": {"source": "web", "attempt": 1},
}

try:
    import msgpack
    HAS_MSGPACK = True
except ImportError:
    HAS_MSGPACK = False
    print("optional: pip install msgpack")



## Decision sketch (from the article)

For each scenario, answer three questions in order:

1. Do people need to read or edit the payload on the wire?
2. Do you need a shared interface definition (an IDL) and multi-language stubs?
3. Is the data staying inside one fully trusted runtime only?

You should see different answers for a public REST body, an internal multi-language RPC call,
a private cache used by a single service binary, and a queue shared by several services.
The point is to practice the **order** of the questions, not to memorize library names.


In [ ]:
def choose_family(human_readable: bool, need_idl: bool, single_trusted_runtime: bool) -> str:
    if human_readable:
        return "Text/JSON family (+ validation layer for real contracts)"
    if need_idl:
        return "Schema-driven (Protobuf/Avro-class)"
    if single_trusted_runtime:
        return "Language-native only inside a hard trust boundary"
    return "Schemaless binary (MessagePack/CBOR-class) + validation at edges"


scenarios = [
    ("Public REST body", True, False, False),
    ("Internal multi-lang RPC", False, True, False),
    ("Redis cache same service only", False, False, True),
    ("Internal queue, multi-service Python+Go", False, False, False),
]
for name, hum, idl, native in scenarios:
    print(f"{name:40} → {choose_family(hum, idl, native)}")



## The same data object, three encodings

Here one logical “data transfer object” (a small API-shaped dictionary) is encoded three ways:

- **JSON** — usually largest, but easy for humans and tools to inspect
- **MessagePack** (if installed) — binary and often smaller, yet map keys still travel with the data
- **A tiny field-number sketch** — denser because names are omitted; this is only a teaching sketch, not full Protocol Buffers

Public APIs often keep JSON even when binary wins on size, because operability matters.
Size alone does not choose the format.


In [ ]:
def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def encode_dto_idl(d: dict) -> bytes:
    # teaching sketch only: 1 request_id, 2 user_id, 3 items as JSON blob (lazy), 4 meta JSON
    out = bytearray()
    rid = d["request_id"].encode()
    out += encode_key(1, 2) + encode_varint(len(rid)) + rid
    out += encode_key(2, 0) + encode_varint(d["user_id"])
    items = json.dumps(d["items"], separators=(",", ":")).encode()
    out += encode_key(3, 2) + encode_varint(len(items)) + items
    meta = json.dumps(d["metadata"], separators=(",", ":")).encode()
    out += encode_key(4, 2) + encode_varint(len(meta)) + meta
    return bytes(out)


j = json.dumps(DTO, separators=(",", ":")).encode()
idl = encode_dto_idl(DTO)
print(f"{'encoding':22} {'nbytes':>6}")
print(f"{'JSON':22} {len(j):6}")
if HAS_MSGPACK:
    mp = msgpack.packb(DTO, use_bin_type=True)
    print(f"{'MessagePack':22} {len(mp):6}")
print(f"{'IDL sketch':22} {len(idl):6}")
print("JSON preview:", j.decode()[:80], "…")



## Validation is still required for JSON

Successfully parsing bytes as JSON only means the punctuation is legal.
It does not mean required fields are present or that types are correct.

The cell checks a good payload and a bad one.
Schemaless and text formats need an **external** contract—OpenAPI, JSON Schema, or typed request models in your framework.
Schema-driven formats move some of that checking into a shared definition; they do not remove the need for clear contracts.


In [ ]:
REQUIRED = {"request_id", "user_id", "items"}


def validate_dto(d: dict) -> list:
    errs = []
    for k in REQUIRED:
        if k not in d:
            errs.append(f"missing {k}")
    if "user_id" in d and not isinstance(d["user_id"], int):
        errs.append("user_id must be int")
    if "items" in d and not isinstance(d["items"], list):
        errs.append("items must be list")
    return errs


good = json.loads(j)
bad = {"user_id": "42"}
assert validate_dto(good) == []
assert validate_dto(bad)
print("OK good:", validate_dto(good))
print("OK bad:", validate_dto(bad))



## Takeaways

1. Start from the situation: human-readable wire, need for a shared IDL, then trust boundary.
2. JSON favors public debuggability; binary favors density—always under your real constraints.
3. Language-native formats (such as pickle) are not a strategy for interchange across services or languages.

**Next:** [Data science lab](./data_science_perspective.ipynb) ·
[Serialization 201](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/)
